# Demo · Anonimización de historias clínicas colombianas

Carga el sistema final exportado por `01_entrenamiento_y_exportacion.ipynb`
(BETO adaptado + reglas colombianas + postproceso y umbral) y lo usa con texto nuevo.
**No entrena nada.**

1. **Celda 1:** instala lo que falte y carga el modelo desde Google Drive
   (`MyDrive/meddocan-anon/export/anonimizador_beto_co`) o desde el zip subido a Colab.
2. **Celda 2:** define `anonimizar()` con las mismas reglas y el mismo postproceso del entrenamiento.
3. **Celda 3:** cambia `TEXTO` (o elige una de las notas de ejemplo) y ejecuta.

Los datos personales de las notas de ejemplo son inventados.

In [1]:
# ===========================================================================
# Celda 1 · Instalar y cargar el sistema exportado
# ===========================================================================
import importlib.util
import json
import re
import subprocess
import sys
import time
import zipfile
from pathlib import Path

if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.46"], check=True)

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

NOMBRE_EXPORT = "anonimizador_beto_co"
CARPETA_DRIVE = Path("/content/drive/MyDrive/meddocan-anon/export")


def buscar_export():
    """Busca la carpeta exportada; si solo hay zip, lo descomprime en /content."""
    carpetas = [CARPETA_DRIVE / NOMBRE_EXPORT, Path("/content") / NOMBRE_EXPORT,
                Path("export") / NOMBRE_EXPORT, Path(NOMBRE_EXPORT)]
    zips = [Path("/content") / f"{NOMBRE_EXPORT}.zip", Path(f"{NOMBRE_EXPORT}.zip"),
            CARPETA_DRIVE / f"{NOMBRE_EXPORT}.zip"]
    for c in carpetas:
        if (c / "config_sistema.json").exists():
            return c
    for z in zips:
        if z.exists():
            destino = Path("/content" if Path("/content").exists() else ".") / NOMBRE_EXPORT
            zipfile.ZipFile(z).extractall(destino)
            return destino
    return None


EXPORT = buscar_export()
if EXPORT is None:
    try:                                   # en Colab: montamos Drive y volvemos a buscar
        from google.colab import drive
        drive.mount("/content/drive")
        EXPORT = buscar_export()
    except ImportError:
        pass
assert EXPORT is not None, (
    f"No se encontró {NOMBRE_EXPORT}. Ejecuta la Caja 10 del notebook de entrenamiento "
    f"o sube {NOMBRE_EXPORT}.zip a Colab.")

CONFIG = json.loads((EXPORT / "config_sistema.json").read_text(encoding="utf-8"))
REGLAS_JSON = json.loads((EXPORT / "reglas.json").read_text(encoding="utf-8"))

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(EXPORT / "modelo")
modelo = AutoModelForTokenClassification.from_pretrained(EXPORT / "modelo").to(DISPOSITIVO).eval()

MAX_LEN, STRIDE, UMBRAL = CONFIG["max_len"], CONFIG["stride"], CONFIG["umbral"]
label2id = CONFIG["label2id"]
id2label = {int(k): v for k, v in CONFIG["id2label"].items()}
GRUPOS = CONFIG["grupos"]

print(f"Sistema: {CONFIG['sistema']}  (exportado {CONFIG['fecha_exportacion']})")
print(f"Carpeta: {EXPORT} | Dispositivo: {DISPOSITIVO} | Umbral: {UMBRAL}")
if CONFIG.get("metricas"):
    display(pd.Series(CONFIG["metricas"], name="valor").to_frame().round(3))

Mounted at /content/drive


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

Sistema: BETO adaptado + reglas + mejoras  (exportado 2026-11-24T05:42:16)
Carpeta: /content/drive/MyDrive/meddocan-anon/export/anonimizador_beto_co | Dispositivo: cpu | Umbral: 0.9


,valor
MEDDOCAN F1,0.962
MEDDOCAN F1 macro,0.809
MEDDOCAN F1 por grupo,0.964
Colombia F1 (con tipo),0.715
Colombia F1 sin tipo,0.870
Fuga crítica (%),2.174
Docs sin fuga (%),88.750


In [2]:
# ===========================================================================
# Celda 2 · anonimizar(): modelo + reglas colombianas + postproceso
# Las funciones son las mismas del notebook de entrenamiento (Cajas 6, 8 y 9);
# las reglas se compilan desde reglas.json para que sean idénticas.
# ===========================================================================
REGLAS_CO_V2 = [(r["tipo"], re.compile(r["patron"], r["flags"])) for r in REGLAS_JSON["reglas_co_v2"]]
CONTEXTO_TIPO = [(re.compile(r["patron"], r["flags"]), r["tipo"]) for r in REGLAS_JSON["contexto_tipo"]]
TIPOS_CORREGIBLES = set(REGLAS_JSON["tipos_corregibles"])
PROPAGABLES = set(REGLAS_JSON["propagables"])
SEXO_VALIDOS = set(REGLAS_JSON["sexo_validos"])
MESES = REGLAS_JSON["meses"]
ID_O = label2id["O"]


# --- Caja 6: de etiquetas BIO a spans y limpieza ------------------------------
def spans_de_etiquetas(offsets, tags):
    """Reconstruye entidades a partir de la secuencia B-/I-/O."""
    spans, actual = [], None
    for (ini, fin), t in zip(offsets, tags):
        etiqueta = id2label[int(t)] if t != -100 else "O"
        if etiqueta == "O":
            if actual:
                spans.append(actual)
                actual = None
            continue

        prefijo, tipo = etiqueta.split("-", 1)
        if prefijo == "B" or actual is None or actual["type"] != tipo:
            if actual:
                spans.append(actual)
            actual = {"start": ini, "end": fin, "type": tipo}
        else:
            actual["end"] = fin

    if actual:
        spans.append(actual)
    return spans


def recortar(span, texto):
    """Quita espacios y puntuación en los bordes del span."""
    ini, fin = span["start"], span["end"]
    while ini < fin and texto[ini] in " \t\n.,;:)]}\"'":
        ini += 1
    while fin > ini and texto[fin - 1] in " \t\n.,;:([{\"'":
        fin -= 1
    return {"start": ini, "end": fin, "type": span["type"]}


def solapan(a, b):
    """Número de caracteres que comparten dos spans."""
    return max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))


def deduplicar(spans):
    """Elimina la misma entidad repetida en ventanas solapadas."""
    ordenados = sorted(spans, key=lambda s: (s["start"], -(s["end"] - s["start"])))
    limpios = []
    for span in ordenados:
        repetido = any(
            span["type"] == c["type"]
            and solapan(span, c) >= 0.5 * min(span["end"] - span["start"], c["end"] - c["start"])
            for c in limpios
        )
        if not repetido:
            limpios.append(span)
    return limpios


def sin_solapes(spans):
    """Conserva el span más largo cuando dos predicciones se cruzan."""
    ordenados = sorted(spans, key=lambda s: (-(s["end"] - s["start"]), s["start"]))
    limpios = []
    for span in ordenados:
        if not any(solapan(span, c) > 0 for c in limpios):
            limpios.append(span)
    return limpios


def validar_span(span, texto):
    """Descarta fragmentos que claramente no son del tipo predicho."""
    frag = texto[span["start"]:span["end"]].strip()
    low = frag.lower()
    tipo = span["type"]

    if not frag:
        return False
    letras = sum(c.isalpha() for c in frag)
    digitos = sum(c.isdigit() for c in frag)

    if tipo == "SEXO_SUJETO_ASISTENCIA":
        return re.sub(r"\s+", "", low.strip(".")) in SEXO_VALIDOS
    if tipo == "EDAD_SUJETO_ASISTENCIA":
        if digitos >= 1:
            return True
        if any(w in low for w in ("año", "mes", "día", "dia", "semana")):
            return True
        return low in {"adolescente", "recién nacido", "recien nacido",
                       "recién nacida", "recien nacida", "lactante"}
    if tipo == "FECHAS":
        if re.search(MESES, low):
            return True
        if digitos < 4:
            return False
        if any(c in frag for c in "/-."):
            return True
        return bool(re.fullmatch(r"\d{4}", frag)) or "año" in low
    if tipo.startswith("ID_"):
        return digitos >= 2 or letras >= 4
    if tipo == "CORREO_ELECTRONICO":
        return len(frag) >= 4 and (letras >= 2 or "@" in frag)
    if tipo in {"NUMERO_TELEFONO", "NUMERO_FAX"}:
        return digitos >= 6
    if tipo == "TERRITORIO":
        return len(frag) >= 2 and (letras >= 2 or digitos >= 2)
    if tipo == "CALLE":
        return len(frag) >= 2 and (letras >= 1 or digitos >= 2)
    if tipo == "PROFESION":
        return letras >= 3
    return len(frag) >= 2 and letras >= 1


# --- Caja 8: unir modelo y reglas ----------------------------------------------
def fusionar(spans_modelo, spans_reglas, texto):
    """Agrega las detecciones de las reglas sin dejar pedazos visibles."""
    resultado = list(spans_modelo)
    for r in spans_reglas:
        r = recortar(r, texto)
        if r["end"] <= r["start"] or not validar_span(r, texto):
            continue
        cruzadas = [p for p in resultado if solapan(p, r) > 0]
        if not cruzadas:
            resultado.append(r)
            continue
        mayor = max(cruzadas + [r], key=lambda s: s["end"] - s["start"])
        unido = {"start": min(s["start"] for s in cruzadas + [r]),
                 "end": max(s["end"] for s in cruzadas + [r]),
                 "type": mayor["type"]}
        resultado = [p for p in resultado if p not in cruzadas] + [unido]
    return sorted(resultado, key=lambda s: s["start"])


# --- Caja 9: probabilidades, umbral, reglas v2, tipo por contexto y propagación --
def probabilidades(modelo_x, tok, texto):
    """Devuelve, por ventana, los offsets y la probabilidad de cada etiqueta."""
    cod = tok(texto, max_length=MAX_LEN, stride=STRIDE, truncation=True,
              return_overflowing_tokens=True, return_offsets_mapping=True,
              add_special_tokens=True)
    ventanas = []
    modelo_x.eval()
    with torch.no_grad():
        for i in range(len(cod["input_ids"])):
            entrada = {
                "input_ids": torch.tensor([cod["input_ids"][i]]).to(DISPOSITIVO),
                "attention_mask": torch.tensor([cod["attention_mask"][i]]).to(DISPOSITIVO),
            }
            p = torch.softmax(modelo_x(**entrada).logits[0].float(), dim=-1)
            ventanas.append((cod["offset_mapping"][i], p.cpu().numpy().astype(np.float16)))
    return ventanas


def decodificar(ventanas, texto, umbral=0.0):
    """Convierte probabilidades en spans, con el umbral de recall de la Caja 9.2."""
    spans = []
    for offsets, p in ventanas:
        p = p.astype(np.float32)
        etiquetas_v = p.argmax(-1)
        if umbral > 0:
            sin_o = p.copy()
            sin_o[:, ID_O] = -1.0
            cambiar = (etiquetas_v == ID_O) & (p[:, ID_O] < umbral)
            etiquetas_v = np.where(cambiar, sin_o.argmax(-1), etiquetas_v)
        etiquetas_v = np.array([ID_O if e <= s else t
                                for (s, e), t in zip(offsets, etiquetas_v)])
        for span in spans_de_etiquetas(offsets, etiquetas_v):
            span = recortar(span, texto)
            if span["end"] > span["start"] and validar_span(span, texto):
                spans.append(span)
    return sin_solapes(deduplicar(spans))


def spans_reglas_v2(texto):
    spans = []
    for tipo, patron in REGLAS_CO_V2:
        for m in patron.finditer(texto):
            if tipo == "FECHAS" or not (m.groups() and m.group(1)):
                ini, fin = m.span()
            else:
                ini, fin = m.span(1)
            spans.append({"start": ini, "end": fin, "type": tipo})
    return sin_solapes(spans)


def corregir_tipo(spans, texto):
    """Cambia el tipo de un identificador según el rótulo que tiene justo antes."""
    corregidos = []
    for s in spans:
        frag = texto[s["start"]:s["end"]]
        digitos = sum(c.isdigit() for c in frag)
        letras = sum(c.isalpha() for c in frag)
        nuevo = dict(s)
        if s["type"] in TIPOS_CORREGIBLES and digitos >= 4 and letras <= 3:
            antes = texto[max(0, s["start"] - 35):s["start"]]
            for patron, tipo in CONTEXTO_TIPO:
                if patron.search(antes):
                    nuevo["type"] = tipo
                    break
        corregidos.append(nuevo)
    return corregidos


def propagar_entidades(spans, texto):
    """Lo que se detectó una vez se tapa en todo el documento."""
    cadenas = {}
    for s in spans:
        if s["type"] not in PROPAGABLES:
            continue
        frag = texto[s["start"]:s["end"]]
        if len(frag) >= 4:
            cadenas.setdefault(frag, s["type"])
        if s["type"].startswith("NOMBRE"):
            for palabra in re.findall(r"[A-ZÁÉÍÓÚÑ][a-záéíóúñü]{3,}", frag):
                cadenas.setdefault(palabra, s["type"])
    nuevos = list(spans)
    for frag, tipo in cadenas.items():
        for m in re.finditer(rf"(?<!\w){re.escape(frag)}(?!\w)", texto):
            candidato = {"start": m.start(), "end": m.end(), "type": tipo}
            if not any(solapan(candidato, p) > 0 for p in nuevos):
                nuevos.append(candidato)
    return sorted(nuevos, key=lambda s: s["start"])


def sistema(ventanas, texto, reglas=None, umbral=0.0, contexto=False, propagar=False):
    """Une todas las piezas en el orden en que se aplican."""
    spans = decodificar(ventanas, texto, umbral)
    if reglas:
        spans = fusionar(spans, reglas(texto), texto)
    if propagar:
        spans = propagar_entidades(spans, texto)
    if contexto:
        spans = corregir_tipo(spans, texto)
    return spans


# --- Función de la demo ----------------------------------------------------------
def anonimizar(texto):
    """Devuelve el texto anonimizado y la lista de entidades detectadas."""
    opciones = CONFIG["opciones"]
    spans = sistema(probabilidades(modelo, tokenizer, texto), texto,
                    reglas=spans_reglas_v2 if opciones.get("reglas") else None,
                    umbral=UMBRAL, contexto=opciones.get("contexto", False),
                    propagar=opciones.get("propagar", False))
    salida = texto
    for s in sorted(spans, key=lambda s: s["start"], reverse=True):
        salida = salida[:s["start"]] + f"[{GRUPOS.get(s['type'], 'OTRO')}]" + salida[s["end"]:]
    return salida, spans


def mostrar(texto):
    """Imprime el original, las entidades con su tipo y el texto anonimizado."""
    t0 = time.perf_counter()
    salida, spans = anonimizar(texto)
    ms = 1000 * (time.perf_counter() - t0)
    print("TEXTO ORIGINAL\n" + "-" * 80 + "\n" + texto.strip() + "\n")
    print(f"ENTIDADES DETECTADAS ({len(spans)})")
    display(pd.DataFrame([{"texto": texto[s["start"]:s["end"]], "tipo": s["type"],
                           "grupo": GRUPOS.get(s["type"], "OTRO"),
                           "inicio": s["start"], "fin": s["end"]} for s in spans]))
    print("\nTEXTO ANONIMIZADO\n" + "-" * 80 + "\n" + salida.strip())
    print(f"\n({ms:.0f} ms en {DISPOSITIVO})")


anonimizar("Paciente de 40 años.")   # calentamiento: la primera llamada es más lenta
print("anonimizar() lista.")

anonimizar() lista.


In [3]:
# ===========================================================================
# Celda 3 · Probar con una nota (edita TEXTO o elige un ejemplo)
# ===========================================================================
EJEMPLOS = {
    "urgencias": """URGENCIAS – TRIAGE II. Clínica San Rafael de Ibagué, sede Picaleña.
Paciente Luz Dary Cárdenas Ospina, 58 años, sexo F, CC 38.256.914, afiliada a Salud Total EPS.
Traída por su hijo Wilmer Andrés Ospina, celular 316 782 4410. Reside en Calle 60 # 5-32,
barrio Jordán, Ibagué (Tolima). Consulta por dolor torácico opresivo desde las 6:00.
TA 150/95 mmHg, FC 98 lpm. Valorada por la Dra. Paula Andrea Quiroga, RM 73214-11.""",

    "evolucion": """Nota de evolución 03/09/2025. Paciente masculino de 72 años, natural de Sogamoso y
residente en el barrio La Floresta de Medellín, quien cursa cuarto día de hospitalización por
neumonía (J18.9). Vive con su esposa Gladys, quien deja el teléfono 604 448 2731.
Se explica a Rigoberto Mesa Arbeláez el plan de manejo. Historia clínica 20457813.
Firma: Dr. Fabio Ernesto Lozano, internista, RM 1.045.223.""",

    "remision": """Bucaramanga, 18 de octubre de 2025

Señores Nueva EPS, oficina de autorizaciones.
Solicitamos la remisión del paciente Kevin Santiago Rueda Pabón, tarjeta de identidad
1.098.776.543, de 9 años, al Hospital Universitario de Santander para valoración por
cirugía pediátrica. Acudiente: Yamile Pabón (madre), correo yamile.pabon@hotmail.com,
celular +57 318 220 9054. Dirección: Cra. 27 # 45-18, apto 302.
Cordialmente, Dra. Natalia Cristina Herrera, médica general, ingreso n.º 7781204.""",
}

# Cambia la clave del ejemplo o escribe tu propio texto entre las comillas triples.
TEXTO = EJEMPLOS["urgencias"]
# TEXTO = """Escribe aquí una nota clínica..."""

mostrar(TEXTO)

TEXTO ORIGINAL
--------------------------------------------------------------------------------
URGENCIAS – TRIAGE II. Clínica San Rafael de Ibagué, sede Picaleña.
Paciente Luz Dary Cárdenas Ospina, 58 años, sexo F, CC 38.256.914, afiliada a Salud Total EPS.
Traída por su hijo Wilmer Andrés Ospina, celular 316 782 4410. Reside en Calle 60 # 5-32,
barrio Jordán, Ibagué (Tolima). Consulta por dolor torácico opresivo desde las 6:00.
TA 150/95 mmHg, FC 98 lpm. Valorada por la Dra. Paula Andrea Quiroga, RM 73214-11.

ENTIDADES DETECTADAS (20)


,texto,tipo,grupo,inicio,fin
0,Clínica San Rafael,HOSPITAL,LUGAR,23,41
1,Ibagué,TERRITORIO,LUGAR,45,51
2,sede,CALLE,LUGAR,53,57
3,Picaleña,TERRITORIO,LUGAR,58,66
4,Luz,NOMBRE_PERSONAL_SANITARIO,NOMBRE,77,80
5,Dary,NOMBRE_SUJETO_ASISTENCIA,NOMBRE,81,85
6,Cárdenas Os,NOMBRE_PERSONAL_SANITARIO,NOMBRE,86,97
7,pina,NOMBRE_SUJETO_ASISTENCIA,NOMBRE,97,101
8,58 años,EDAD_SUJETO_ASISTENCIA,OTRO,103,110
9,sexo,ID_SUJETO_ASISTENCIA,IDENTIFICACIÓN,112,116



TEXTO ANONIMIZADO
--------------------------------------------------------------------------------
URGENCIAS – TRIAGE II. [LUGAR] de [LUGAR], [LUGAR] [LUGAR].
Paciente [NOMBRE] [NOMBRE] [NOMBRE][NOMBRE], [OTRO], [IDENTIFICACIÓN] F, [IDENTIFICACIÓN], afiliada a [LUGAR] [LUGAR].
Traída por su [NOMBRE], [CONTACTO]. Reside en [LUGAR], [LUGAR] ([LUGAR]). Consulta por dolor torácico opresivo desde las 6:00.
TA 150/95 mmHg, FC 98 lpm. Valorada por la Dra. [NOMBRE], [CONTACTO].

(601 ms en cpu)
